<a href="https://colab.research.google.com/github/Kanchanajaddu/GEN_AI-exercises/blob/main/genaiweek2day1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#given question:
#take 5-10 passages from wikipedia or docs
#generate embeddings with sentence transformers
#run similarity search with faiss/chroma

In [ ]:
passage1="The quick brown fox jumps over the lazy dog. This classic pangram contains every letter of the English alphabet, making it a popular choice for typing tests and font demonstrations. It's a short, memorable sentence that effectively showcases text formatting and legibility. Beyond its practical uses, the phrase has ingrained itself in popular culture as a concise example of linguistic completeness."

In [ ]:
passage2="The vastness of space continues to captivate humanity, with billions of galaxies stretching across the observable universe. Each galaxy, a cosmic island of stars, planets, and nebulae, moves through the void, driven by forces we are only beginning to comprehend. The study of cosmology seeks to unravel these mysteries, exploring the origins, evolution, and ultimate fate of the cosmos."

In [ ]:
passage3="The invention of the printing press by Johannes Gutenberg in the 15th century revolutionized the dissemination of knowledge. Prior to this, books were painstakingly copied by hand, making them rare and expensive. The printing press enabled mass production of texts, drastically lowering their cost and making literacy more accessible to the general population, thus fueling the Renaissance and the Reformation."

In [ ]:
passage4="Photosynthesis is the fundamental biochemical process by which plants, algae, and some bacteria convert light energy into chemical energy. This energy is stored in glucose, a sugar molecule, and is essential for nearly all life on Earth. Through chlorophyll, these organisms absorb sunlight, carbon dioxide, and water to create oxygen as a byproduct, shaping our planet's atmosphere."

In [ ]:
passage5="Artificial intelligence, often abbreviated as AI, refers to the simulation of human intelligence in machines programmed to think like humans and mimic their actions. This broad field encompasses machine learning, deep learning, natural language processing, and computer vision. AI applications are rapidly transforming industries from healthcare to finance, promising both unprecedented efficiency and complex ethical challenges."

In [ ]:
passage6="Plants are multicellular eukaryotes that form the kingdom Plantae. They are characterized by their ability to photosynthesize, a process that converts light energy into chemical energy, using chlorophyll. Essential for life on Earth, plants produce the oxygen we breathe and are the primary producers in most terrestrial food chains. Their diversity ranges from microscopic algae to towering trees, occupying nearly every ecological niche."

In [ ]:
passage7="The exploration of space has been a defining characteristic of human scientific endeavor since the mid-20th century. Satellites orbiting Earth provide invaluable data for communication, navigation, and climate monitoring, while probes venture into the farthest reaches of our solar system and beyond. Manned missions continue to push the boundaries of human presence in space, aiming for destinations like the Moon and Mars, driven by curiosity and the quest for knowledge about our universe."

In [ ]:
#generate embeddings with sentence transformers


In [ ]:
#install sentence transformers
!pip install -U sentence_transformers

In [ ]:
#importing sentence Transformer model
from sentence_transformers import SentenceTransformer

In [ ]:
#all passages set into a list
passages=[passage1,
    passage2,
    passage3,
    passage4,
    passage5,
    passage6,
    passage7]
#loading a pretrained sentence transformer model
model=SentenceTransformer('all-MiniLM-L6-v2')
#generate embeddings
passage_embeddings=model.encode(passages)
print("embeddings shape")
print(passage_embeddings.shape)
print("generated first embedding sample")
#printing the first 5 dimensions of the first embedding
print(passage_embeddings[0][:5])


In [ ]:
#printing the first five dimensions of 5th embedding
print(passage_embeddings[4][:5])

In [ ]:
#run similarity search with faiss/chroma

In [ ]:
# Install faiss-cpu
!pip install faiss-cpu

In [ ]:
# Import faiss and numpy
import faiss
import numpy as np

# Get the dimensionality of the embeddings
dimension = passage_embeddings.shape[1]

# Create a FAISS index
# Here we use IndexFlatL2 for a simple Euclidean distance search
index = faiss.IndexFlatL2(dimension)

# Add the passage embeddings to the index
index.add(np.array(passage_embeddings))

print(f"FAISS index created with {index.ntotal} vectors of dimension {index.d}")

In [ ]:
# Define a query
query = "What is artificial intelligence?"

# Generate embedding for the query
query_embedding = model.encode([query])

# Perform a similarity search
k = 3 # Number of nearest neighbors to retrieve
distances, indices = index.search(np.array(query_embedding), k)

print(f"Query: {query}")
print("\nMost similar passages:")
for i in range(k):
    print(f"- Passage {indices[0][i]+1} (Distance: {distances[0][i]:.4f}): {passages[indices[0][i]]}")

In [ ]:
# Install chromadb
!pip install chromadb

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

# Initialize a Chroma client
# For simplicity, we'll use an in-memory client
client = chromadb.Client()

# Create a collection. If it already exists, get it, otherwise create it.
# Catching the specific NotFoundError from chromadb, or a general Exception if it's not directly accessible.
try:
    collection = client.get_collection(name="passages")
except Exception as e: # Catch a more general exception to avoid AttributeError
    # Check if the error message indicates a collection not found error
    if "Collection [passages] does not exist" in str(e): # This matches the NotFoundError message
        collection = client.create_collection(name="passages")
    else:
        raise e # Re-raise other unexpected exceptions

# Prepare documents and their IDs for Chroma
documents = passages
# Chroma requires string IDs, so we'll use indices as IDs
ids = [f"doc{i}" for i in range(len(passages))]

# Add the passages and their embeddings to the collection
# Chroma can take embeddings directly if you provide them
collection.add(
    embeddings=passage_embeddings.tolist(), # Convert numpy array to list
    documents=documents,
    ids=ids
)

print(f"Chroma collection created with {collection.count()} documents.")

In [ ]:
# Define a query
query_chroma = "What is artificial intelligence?"

# Perform a similarity search using Chroma
k_chroma = 3 # Number of nearest neighbors to retrieve

results = collection.query(
    query_embeddings=model.encode([query_chroma]).tolist(), # Encode query and convert to list
    n_results=k_chroma
)

print(f"Query: {query_chroma}")
print("\nMost similar passages (Chroma):")
# The results are structured, extract documents and distances
# Chroma returns similarity scores, lower is better (closer to 0 for perfect match) for normalized embeddings
# Or higher for dot product (depending on internal implementation and normalisation)
# We'll print distances as provided by Chroma

if results and results['documents'] and results['distances']:
    for i in range(len(results['documents'][0])):
        doc_content = results['documents'][0][i]
        distance = results['distances'][0][i]
        # Find the original passage index to reference the 'Passage X' style
        original_passage_index = passages.index(doc_content) + 1
        print(f"- Passage {original_passage_index} (Distance: {distance:.4f}): {doc_content}")
else:
    print("No results found.")

In [ ]:
#similarity between two embeddings

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Choose two embeddings for comparison (e.g., passage1 and passage5)
embedding4 = passage_embeddings[3] # Embedding for 'The quick brown fox...'
embedding6 = passage_embeddings[5] # Embedding for 'Artificial intelligence...'

# Reshape embeddings for cosine_similarity if they are 1D arrays
embedding4_reshaped = embedding4.reshape(1, -1)
embedding6_reshaped = embedding6.reshape(1, -1)

# Calculate cosine similarity
similarity_score = cosine_similarity(embedding4_reshaped, embedding6_reshaped)[0][0]

print(f"Similarity between Passage 4 and Passage 6: {similarity_score:.4f}")

# Choose two embeddings that are expected to be more similar (e.g., passage5 and passage7 are related to AI/Space)
embedding5 = passage_embeddings[4] # Embedding for 'Artificial intelligence...'
embedding7 = passage_embeddings[6] # Embedding for 'The exploration of space...'

embedding5_reshaped = embedding5.reshape(1, -1)
embedding7_reshaped = embedding7.reshape(1, -1)

similarity_score_2 = cosine_similarity(embedding5_reshaped, embedding7_reshaped)[0][0]

print(f"Similarity between Passage 5 and Passage 7: {similarity_score_2:.4f}")
